# Day 2: Parameter-Efficient Fine-Tuning (LoRA / QLoRA)

This notebook demonstrates fine-tuning an LLM using QLoRA with 4-bit quantization.
It implements memory-saving tricks like 4-bit loading, gradient checkpointing, and mixed precision.

**Target Model:** TinyLlama-1.1B 

**Dataset:** Coding Instruction Dataset

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.0 MB/s eta 0:00:00


In [17]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

In [18]:
# Defining model and dataset
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 1. Loading the dataset
try:
    dataset = load_dataset('json', data_files='train_clean.jsonl', split='train')
    print(f"Loaded {len(dataset)} samples from day 1 dataset.")
except Exception as e:
    print("Error loading dataset. Falling back to Guanaco for demo.")
    dataset = load_dataset("timdettmers/openassistant-guanaco", split="train").select(range(500))

# 2. Formatting function for Instruction Tuning
def format_instruction(sample):
    instruction = f"### Instruction:\n{sample['instruction']}"
    input_val = f"### Input:\n{sample['input']}" if sample.get('input') else ""
    output = f"### Response:\n{sample['output']}"

    parts = [p for p in [instruction, input_val, output] if p]
    return "\n\n".join(parts)

print("Sample Formatted Input:")
print(format_instruction(dataset[0]))

Loaded 1000 samples from day 1 dataset.
Sample Formatted Input:
### Instruction:
[Python] Technical Extraction: Identify the rate limiting parameters (count and period). (ID-291)

### Input:
// Application Domain: a global content delivery network
class API:
    @auth_required
    @rate_limit(count=100, period='1m')
    def fetch(self): pass

### Response:
count: 100, period: 1m


In [27]:
# 2. Loading model with explicit float16 (Standardized for T4)
compute_dtype = torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16  # Force weights to float16
)

model.config.torch_dtype = torch.float16
model.config.use_cache = False


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [28]:
# 3. Preparation for k-bit training
model = prepare_model_for_kbit_training(model)

for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

print("Model fully sanitized: All BFloat16 tensors converted to Float16.")


Model fully sanitized: All BFloat16 tensors converted to Float16.


In [29]:
# 4. Setup LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
print("Model fully sanitized and ready for T4 GPU!")

Model fully sanitized and ready for T4 GPU!


In [ ]:
# 5. Training Configuration
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="./adapters",
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    formatting_func=format_instruction,
    processing_class=tokenizer,
    args=sft_config,
)

# 6. Execute Training
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.162219
20,1.041215
30,0.427378
40,0.194971
50,0.145248
60,0.154011
70,0.138607
80,0.133080
90,0.139600
100,0.131941


TrainOutput(global_step=750, training_loss=0.16454931036631265, metrics={'train_runtime': 598.081, 'train_samples_per_second': 5.016, 'train_steps_per_second': 1.254, 'total_flos': 2676383938363392.0, 'train_loss': 0.16454931036631265})

In [32]:
# 7. Save Adapter Weights
trainer.model.save_pretrained("./adapters/polyglot_lora_adapter")
print("Training complete! Adapter saved in ./adapters/polyglot_lora_adapter")

Training complete! Adapter saved in ./adapters/polyglot_lora_adapter
